<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Validación de Clusters, Selección de K y Benchmark
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 10
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/04_Validacion_Seleccion_K_y_Benchmark_Comparativo.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request
import warnings
warnings.filterwarnings('ignore')

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (8.5, 4.5)
plt.rcParams['font.size'] = 10

def load_dataset(filename, module_folder="10 - Clustering"):
    local_path = os.path.join(os.getcwd(), "data", filename)
    if os.path.exists(local_path):
        return local_path
    
    parent_path = os.path.join(os.getcwd(), "..", module_folder, "data", filename)
    if os.path.exists(parent_path):
        return parent_path

    raw_url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/Data%20Science%20programming/{module_folder.replace(' ', '%20')}/data/{filename}"
    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    if not os.path.exists(target_path):
        urllib.request.urlretrieve(raw_url, target_path)
    return target_path

print("🚀 Entorno configurado exitosamente para el Módulo 10: Clustering.")


---
### 1. Validación de Agrupamiento: ¿Cómo evaluar sin etiquetas reales? 📐

Dado que el clustering es no supervisado, no disponemos de una verdad terreno (*Ground Truth*). La evaluación cuantitativa se realiza mediante **Métricas de Validación Interna**, las cuales miden dos propiedades fundamentales:
1. **Compacidad Intra-Cluster (*Compactness*):** Qué tan cerca están los puntos de su propio centroide/grupo.
2. **Separación Inter-Cluster (*Separation*):** Qué tan distantes y aislados están los clusters entre sí.


---
### 2. Métricas de Validación Interna 🧮

#### A. Coeficiente de Silueta (*Silhouette Score*)
Para cada observación $i$:
* $a(i)$: Distancia promedio desde $i$ hasta todos los demás puntos de su mismo cluster (*cohesión intra-cluster*).
<div align="center"><img src="images/silhouette_intra.png" width="230"/></div>
* $b(i)$: Distancia promedio desde $i$ hasta todos los puntos del cluster más cercano (*separación inter-cluster*).
<div align="center"><img src="images/silhouette_outer.png" width="230"/></div>

La silueta individual $s(i)$ se define como:
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))} \in [-1, 1]$$
<div align="center"><img src="images/silhouette_eq.png" width="280"/></div>

* $s(i) \approx +1$: La observación está perfectamente asignada a su cluster.
* $s(i) \approx 0$: La observación está en la frontera de decisión entre dos clusters.
* $s(i) < 0$: La observación probablemente fue asignada al cluster incorrecto.

#### B. Índice de Davies-Bouldin
Mide la similitud promedio entre cada cluster y su cluster más similar:
$$DB = \frac{1}{k} \sum_{i=1}^k \max_{j 
eq i} \left( \frac{s_i + s_j}{d(\mu_i, \mu_j)} ight)$$
* **Menor valor = Mejor partición** (clusters más compactos y separados).

#### C. Índice de Calinski-Harabasz (Variance Ratio Criterion)
Razón entre la dispersión inter-cluster y la dispersión intra-cluster:
$$CH = \frac{\text{Tr}(B_k)}{\text{Tr}(W_k)} \times \frac{n - k}{k - 1}$$
* **Mayor valor = Mejor partición**.


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

path_data = load_dataset('mall_customers.csv', '10 - Clustering')
df = pd.read_csv(path_data)
X = StandardScaler().fit_transform(df[['Annual_Income_k', 'Spending_Score']].values)

# Evaluar K-Means para distintos valores de k
resultados = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X)
    
    sil = silhouette_score(X, labels)
    db = davies_bouldin_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    resultados.append({'k': k, 'Inercia': km.inertia_, 'Silhouette': sil, 'Davies-Bouldin': db, 'Calinski-Harabasz': ch})

df_res = pd.DataFrame(resultados)
print("--- Métricas de Validación Interna para K-Means ---")
df_res.round(3)


---
### 3. Métodos Gráficos de Selección de $k$ Óptimo 🎯


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# 1. Método del Codo (Elbow Method)
axes[0].plot(df_res['k'], df_res['Inercia'], marker='o', color='#0284c7', linewidth=2)
axes[0].set_title('Método del Codo (Inercia WCSS)', fontweight='bold')
axes[0].set_xlabel('Número de Clusters (k)')
axes[0].set_ylabel('Inercia (WCSS)')
axes[0].axvline(x=5, color='r', linestyle='--', label='Codo Óptimo k=5')
axes[0].legend()

# 2. Coeficiente de Silueta
axes[1].plot(df_res['k'], df_res['Silhouette'], marker='s', color='#10b981', linewidth=2)
axes[1].set_title('Coeficiente de Silueta Global vs k', fontweight='bold')
axes[1].set_xlabel('Número de Clusters (k)')
axes[1].set_ylabel('Silhouette Score Promedio')
axes[1].axvline(x=5, color='r', linestyle='--', label='Pico Máximo k=5')
axes[1].legend()

plt.tight_layout()
plt.show()


---
### 4. Gran Benchmark Comparativo de Algoritmos de Clustering 🏁

A continuación comparamos **K-Means**, **Agglomerative Clustering (HAC)** y **DBSCAN** sobre 4 geometrías sintéticas clásicas:
1. Blobs isotrópicos esféricos.
2. Círculos concéntricos entrelazados.
3. Semilunas continuas (*Moons*).
4. Grupos con varianzas anisotrópicas elongadas.


In [ ]:
from sklearn import datasets
from sklearn.cluster import AgglomerativeClustering, DBSCAN

np.random.seed(42)
n_samples = 300

# 1. Datasets sintéticos
blobs = datasets.make_blobs(n_samples=n_samples, random_state=42)
circles = datasets.make_circles(n_samples=n_samples, factor=0.5, noise=0.05, random_state=42)
moons = datasets.make_moons(n_samples=n_samples, noise=0.05, random_state=42)
X_aniso, _ = datasets.make_blobs(n_samples=n_samples, random_state=42)
transformation = [[0.6, -0.6], [-0.4, 0.8]]
aniso = (np.dot(X_aniso, transformation), None)

conjuntos = [('Blobs Esféricos', blobs[0]), ('Círculos Concéntricos', circles[0]), 
             ('Semilunas', moons[0]), ('Anisotrópicos', aniso[0])]

fig, axes = plt.subplots(4, 3, figsize=(12, 12))
nombres_alg = ['K-Means', 'HAC (Ward/Single)', 'DBSCAN']

for row_idx, (nombre_ds, X_data) in enumerate(conjuntos):
    X_d = StandardScaler().fit_transform(X_data)
    
    # K-Means
    km_lbl = KMeans(n_clusters=2 if row_idx > 0 else 3, random_state=42).fit_predict(X_d)
    axes[row_idx, 0].scatter(X_d[:, 0], X_d[:, 1], c=km_lbl, cmap='tab10', s=25, alpha=0.8)
    if row_idx == 0: axes[row_idx, 0].set_title('K-Means', fontweight='bold', fontsize=12)
    axes[row_idx, 0].set_ylabel(nombre_ds, fontweight='bold', fontsize=11)
    
    # HAC
    link = 'single' if row_idx in [1, 2] else 'ward'
    hac_lbl = AgglomerativeClustering(n_clusters=2 if row_idx > 0 else 3, linkage=link).fit_predict(X_d)
    axes[row_idx, 1].scatter(X_d[:, 0], X_d[:, 1], c=hac_lbl, cmap='tab10', s=25, alpha=0.8)
    if row_idx == 0: axes[row_idx, 1].set_title(f'HAC ({link})', fontweight='bold', fontsize=12)
    
    # DBSCAN
    eps_val = 0.25 if row_idx in [1, 2] else 0.35
    db_lbl = DBSCAN(eps=eps_val, min_samples=5).fit_predict(X_d)
    axes[row_idx, 2].scatter(X_d[:, 0], X_d[:, 1], c=db_lbl, cmap='tab10', s=25, alpha=0.8)
    if row_idx == 0: axes[row_idx, 2].set_title('DBSCAN', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()


---
##### 🛠️ Práctica 5: Selección de Modelo Óptimo con Silhouette Analysis

**Reto:**
1. Carga `mall_customers.csv` y utiliza las características `Age`, `Annual_Income_k` y `Spending_Score`.
2. Estandariza los datos con `StandardScaler`.
3. Ajusta `KMeans(n_clusters=5)` y `AgglomerativeClustering(n_clusters=5, linkage='ward')`.
4. Calcula y compara el `silhouette_score` y `davies_bouldin_score` de ambos modelos.
5. Justifica cuál de los dos algoritmos generó una segmentación cuantitativamente superior.


In [ ]:
# =========================================================================
# TU SOLUCIÓN: Práctica 5 - Benchmark y Selección con Silueta
# =========================================================================

# 1. Variables y escalado
# X_eval = StandardScaler().fit_transform(df[['Age', 'Annual_Income_k', 'Spending_Score']])

# 2. Ajuste de modelos
# ...


<details>
<summary><b>💡 Haz clic aquí para ver la Solución Paso a Paso y Explicación</b></summary>
<br>

```python
# 1. Preparación de datos
X_eval = StandardScaler().fit_transform(df[['Age', 'Annual_Income_k', 'Spending_Score']])

# 2. Modelos con k=5
km_5 = KMeans(n_clusters=5, init='k-means++', random_state=42).fit_predict(X_eval)
hac_5 = AgglomerativeClustering(n_clusters=5, linkage='ward').fit_predict(X_eval)

# 3. Métricas
sil_km = silhouette_score(X_eval, km_5)
sil_hac = silhouette_score(X_eval, hac_5)

db_km = davies_bouldin_score(X_eval, km_5)
db_hac = davies_bouldin_score(X_eval, hac_5)

print(f"K-Means (k=5)  -> Silueta: {sil_km:.4f} | Davies-Bouldin: {db_km:.4f}")
print(f"HAC Ward (k=5) -> Silueta: {sil_hac:.4f} | Davies-Bouldin: {db_hac:.4f}")

if sil_km > sil_hac:
    print("\n🏆 Conclusión: K-Means logró una mayor cohesión y separación inter-cluster.")
else:
    print("\n🏆 Conclusión: HAC Ward produjo una mejor partición geométrica.")
```
</details>


---
### 5. Resumen y Conclusiones del Cuaderno 04 📌

1. **Validación Sin Etiquetas:** El Coeficiente de Silueta (máximo deseado) y Davies-Bouldin (mínimo deseado) son las herramientas estándar para auditar la calidad del clustering.
2. **Método del Codo vs Silueta:** El codo detecta la pérdida de rendimiento marginal, mientras que el pico de silueta valida directamente la compacidad y separación.
3. **Selección de Algoritmo:**
   * **K-Means:** Óptimo para datos tabulares masivos y clusters esféricos.
   * **HAC:** Ideal para entender jerarquías y datasets medianos.
   * **DBSCAN:** Insuperable para detección de anomalías y formas no convexas.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>
